# Notebook 05 — Visualizations

**7 charts answering the 7 research questions.**

Each cell contains one chart followed by a design-justification markdown cell citing course concepts (Tufte, Stephen Few, Gestalt, preattentive attributes).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
from scipy.ndimage import uniform_filter1d
import os

PROC_DIR   = '../data/processed'
CHARTS_DIR = '../charts'
os.makedirs(CHARTS_DIR, exist_ok=True)

master = pd.read_csv(os.path.join(PROC_DIR, 'master.csv'), low_memory=False)
print(f'Master: {master.shape}')

# ── Global style (applied once, affects all charts) ────────────────────────
plt.rcParams.update({
    'figure.facecolor':    'white',
    'axes.facecolor':      'white',
    'axes.spines.top':     False,
    'axes.spines.right':   False,
    'axes.spines.left':    True,
    'axes.spines.bottom':  True,
    'axes.grid':           True,
    'grid.color':          '#E2E8F0',
    'grid.linewidth':      0.6,
    'grid.axis':           'y',
    'font.family':         'DejaVu Sans',
    'font.size':           11,
    'axes.titlesize':      13,
    'axes.titleweight':    'bold',
    'axes.labelsize':      11,
    'xtick.labelsize':     10,
    'ytick.labelsize':     10,
    'legend.frameon':      False,
    'legend.fontsize':     10,
})

# Project palette (2 main colours + neutral)
C_MALE   = '#028090'   # teal
C_FEMALE = '#F96167'   # coral
C_BOTH   = '#2D3748'   # dark slate
C_LIGHT  = '#A0AEC0'   # grey for low-confidence / background

print('Libraries loaded. Global style applied.')

---
## Visualization 1 — Age vs. Finish Time
**Research question:** Does age determine peak running performance? At what age are runners fastest?

In [ ]:
df1 = master.dropna(subset=['age', 'finish_min', 'gender']).copy()
df1 = df1[df1['gender'].isin(['M', 'F'])]

# Subsample for performance if very large (keep all data for stats, sample for scatter)
SCATTER_SAMPLE = 50_000
df1_scatter = df1.sample(min(SCATTER_SAMPLE, len(df1)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))

for gender, color, label in [('M', C_MALE, 'Men'), ('F', C_FEMALE, 'Women')]:
    sub = df1_scatter[df1_scatter['gender'] == gender]
    ax.scatter(sub['age'], sub['finish_min'],
               alpha=0.07, color=color, s=6, rasterized=True)

    # LOESS-style smoothing: bin by age, take median, smooth
    full = df1[df1['gender'] == gender]
    age_bins = pd.cut(full['age'], bins=range(15, 86, 2))
    trend = full.groupby(age_bins, observed=True)['finish_min'].median().reset_index()
    trend['age_mid'] = trend['age'].apply(lambda x: x.mid)
    trend = trend.dropna()
    smoothed = uniform_filter1d(trend['finish_min'].values, size=3)
    ax.plot(trend['age_mid'], smoothed, color=color, lw=2.5, label=label)

# Annotate peak age
for gender, color in [('M', C_MALE), ('F', C_FEMALE)]:
    full = df1[df1['gender'] == gender]
    age_bins = pd.cut(full['age'], bins=range(15, 86, 2))
    trend = full.groupby(age_bins, observed=True)['finish_min'].median().reset_index()
    trend['age_mid'] = trend['age'].apply(lambda x: x.mid)
    trend = trend.dropna()
    peak_row = trend.loc[trend['finish_min'].idxmin()]
    ax.annotate(
        f"peak ~{int(peak_row['age_mid'])}y",
        xy=(peak_row['age_mid'], peak_row['finish_min']),
        xytext=(peak_row['age_mid'] + 3, peak_row['finish_min'] - 8),
        color=color, fontsize=9,
        arrowprops=dict(arrowstyle='->', color=color, lw=1)
    )

ax.set_title('Peak Marathon Performance Occurs in the Early 30s')
ax.set_xlabel('Runner Age (years)')
ax.set_ylabel('Finish Time (minutes)')
ax.legend()
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x//60)}h{int(x%60):02d}m'))

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '01_age_vs_time.png'), dpi=150, bbox_inches='tight')
plt.show()

**Design justification (Viz 1):**
- *Preattentive attribute — color hue* (Lezione 3a): two colors (teal/coral) let the viewer distinguish male vs. female series in <250ms without reading the legend.
- *Alpha transparency* (alpha=0.07): with hundreds of thousands of points, full opacity would produce an unreadable blob. Low alpha reveals density patterns (data-ink ratio).
- *LOESS-style median trend*: avoids forcing a linear model on a clearly non-linear age–performance curve (Stephen Few: match chart type to data shape).
- *Direct annotation of peak age*: eliminates the need for the viewer to identify the minimum manually — reduces working memory load (Lezione 3a: iconic/working memory).
- *Formatted Y-axis* (H:MMm): raw minutes are harder to interpret than clock time.

---
## Visualization 2 — Pacing Strategy by Gender
**Research question:** Do men and women pace differently throughout a marathon (first vs second half splits)?

In [ ]:
df2 = master.dropna(subset=['half_min', 'finish_min', 'gender']).copy()
df2 = df2[df2['gender'].isin(['M', 'F'])]

# Pace in min/km for each half
HALF_DIST_KM = 21.0975
df2['pace_h1'] = df2['half_min'] / HALF_DIST_KM
df2['pace_h2'] = (df2['finish_min'] - df2['half_min']) / HALF_DIST_KM

pacing = df2.groupby('gender')[['pace_h1', 'pace_h2']].agg(['mean', 'sem']).reset_index()

fig, ax = plt.subplots(figsize=(7, 5))

x = [0, 1]
x_labels = ['First half', 'Second half']

for gender, color, label in [('M', C_MALE, 'Men'), ('F', C_FEMALE, 'Women')]:
    row = pacing[pacing['gender'] == gender].iloc[0]
    y    = [row[('pace_h1', 'mean')], row[('pace_h2', 'mean')]]
    yerr = [row[('pace_h1', 'sem')],  row[('pace_h2', 'sem')]]
    ax.errorbar(x, y, yerr=yerr, marker='o', markersize=8,
                color=color, lw=2.5, capsize=4, label=label)
    # Direct label at second half endpoint
    slowdown = y[1] - y[0]
    ax.annotate(
        f'{label}\n+{slowdown:.2f} min/km',
        xy=(1, y[1]),
        xytext=(1.08, y[1]),
        color=color, fontsize=9, va='center'
    )

ax.set_xlim(-0.3, 1.6)
ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=11)
ax.set_title('Women Maintain Pace Better in the Second Half')
ax.set_ylabel('Average Pace (min/km)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'{int(y)}:{int((y%1)*60):02d}'))
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '02_pacing_gender.png'), dpi=150, bbox_inches='tight')
plt.show()

**Design justification (Viz 2):**
- *Gestalt continuity*: the line connecting two data points implies a continuous pacing process, which is the right metaphor for a race split.
- *Error bars (SEM)*: show uncertainty without obscuring the trend — graphical integrity (Tufte: show uncertainty).
- *Direct labeling instead of legend*: annotating each line at its endpoint uses spatial proximity (Gestalt proximity) to link label to data and reduces eye movement.
- *No fill / minimal gridlines*: maximizes data-ink ratio (Tufte).

---
## Visualization 3 — Temperature Effect on Finish Times
**Research question:** How much does race-day temperature affect finish times?

In [ ]:
df3 = master.dropna(subset=['temp_mean_c', 'finish_min']).copy()

# Use median finish time per (city, race_date) to avoid individual noise
grp = df3.groupby(['temp_mean_c'])['finish_min'].median().reset_index()
# But also keep raw scatter
# Sample raw points
df3_scatter = df3.sample(min(30_000, len(df3)), random_state=42)

# Regression
slope, intercept, r, p, se = stats.linregress(df3['temp_mean_c'], df3['finish_min'])
x_line = np.linspace(df3['temp_mean_c'].min(), df3['temp_mean_c'].max(), 200)
y_line = slope * x_line + intercept

# 95% confidence band
n = len(df3)
x_mean = df3['temp_mean_c'].mean()
ssx = ((df3['temp_mean_c'] - x_mean)**2).sum()
s_err = np.sqrt(((df3['finish_min'] - (intercept + slope * df3['temp_mean_c']))**2).sum() / (n - 2))
ci = stats.t.ppf(0.975, n - 2) * s_err * np.sqrt(1/n + (x_line - x_mean)**2 / ssx)

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(df3_scatter['temp_mean_c'], df3_scatter['finish_min'],
           alpha=0.05, color=C_BOTH, s=5, rasterized=True)
ax.plot(x_line, y_line, color=C_FEMALE, lw=2.5,
        label=f'Linear fit  r={r:.2f}, p={p:.2e}')
ax.fill_between(x_line, y_line - ci, y_line + ci,
                alpha=0.2, color=C_FEMALE, label='95% CI')

# Annotate slope
mid_x = df3['temp_mean_c'].median()
ax.annotate(
    f'+{slope:.1f} min per °C',
    xy=(mid_x, intercept + slope * mid_x),
    xytext=(mid_x - 5, intercept + slope * mid_x + 20),
    color=C_FEMALE, fontsize=10,
    arrowprops=dict(arrowstyle='->', color=C_FEMALE, lw=1)
)

ax.set_title('Hotter Race Days Produce Slower Finish Times')
ax.set_xlabel('Mean Temperature on Race Day (°C)')
ax.set_ylabel('Finish Time (minutes)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x//60)}h{int(x%60):02d}m'))
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '03_temp_vs_time.png'), dpi=150, bbox_inches='tight')
plt.show()

**Design justification (Viz 3):**
- *Confidence band*: makes uncertainty explicit — graphical integrity (Tufte: represent numbers proportionally to their quantity; a regression without uncertainty is overconfident).
- *Annotated slope*: the title and annotation together state the finding directly, moving the viewer from sensation → cognition without stopping at the perception stage.
- *Low-alpha scatter*: hundreds of thousands of individual data points underlie the trend. The alpha layer preserves them honestly without dominating (data-ink ratio).
- *Single color for scatter + regression*: no categorical split — the question is about temperature, not demographics. Adding gender color here would invite the wrong comparison.

---
## Visualization 4 — Country Performance Map (Choropleth)
**Research question:** Which countries produce the fastest recreational runners?

In [ ]:
# Try geopandas + plotly. Fall back to a sorted bar chart if geopandas unavailable.
try:
    import geopandas as gpd
    import plotly.express as px
    HAS_GEO = True
except ImportError:
    HAS_GEO = False
    print('geopandas/plotly not installed — falling back to bar chart')

country_col = 'country' if 'country' in master.columns else None

if country_col:
    df4 = master.dropna(subset=[country_col, 'finish_min']).copy()
    country_stats = (
        df4.groupby(country_col)['finish_min']
        .agg(median_finish='median', n_runners='count')
        .reset_index()
    )
    # Only keep countries with ≥ 20 runners (statistical reliability)
    country_stats = country_stats[country_stats['n_runners'] >= 20]
    print(f'Countries with ≥20 runners: {len(country_stats)}')
    print(country_stats.sort_values('median_finish').head(10))
else:
    print('No country column found in master — skipping visualization 4')
    country_stats = None

In [ ]:
if country_stats is not None and HAS_GEO:
    fig_plotly = px.choropleth(
        country_stats,
        locations=country_col,
        locationmode='country names',
        color='median_finish',
        color_continuous_scale='Blues_r',  # reversed: darker = faster (lower time)
        hover_data={'n_runners': True, 'median_finish': ':.0f'},
        title='Median Marathon Finish Time by Runner Nationality (minutes)',
        labels={'median_finish': 'Median finish (min)', country_col: 'Country'},
    )
    fig_plotly.update_layout(
        font_family='DejaVu Sans',
        coloraxis_colorbar=dict(title='Finish time (min)'),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    fig_plotly.write_image(os.path.join(CHARTS_DIR, '04_choropleth_country.png'), scale=2)
    fig_plotly.show()

elif country_stats is not None:
    # Bar chart fallback
    top = country_stats.sort_values('median_finish').head(20)
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(top[country_col], top['median_finish'], color=C_MALE)
    ax.set_title('Top 20 Countries by Median Marathon Finish Time')
    ax.set_xlabel('Median Finish Time (minutes)')
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x//60)}h{int(x%60):02d}m'))
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS_DIR, '04_choropleth_country.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Visualization 4 skipped — no country data available.')

**Design justification (Viz 4):**
- *Sequential colormap (Blues_r)*: continuous variable (median finish time) maps to a single-hue gradient. Never rainbow — it introduces false perceptual categories in continuous data (Lezione 3b colormap section).
- *Reversed scale* (darker = faster): culturally, darker = more intense = better performance. Avoids counter-intuitive reading.
- *Filter ≥20 runners*: countries with very few runners have unreliable medians. Showing grey for low-confidence regions is graphical integrity (Tufte: avoid misleading precision).
- *Geographic context*: choropleth leverages the viewer's pre-existing cognitive map of world geography — faster than reading a ranked list for spatial patterns (Lezione 3a: long-term memory schemas).

---
## Visualization 5 — Performance Trend Over Time (2000–2019)
**Research question:** Have recreational marathon runners gotten faster or slower over two decades, and how has participation changed?


In [ ]:
df5 = master.dropna(subset=['year', 'finish_min', 'gender']).copy()
df5 = df5[df5['gender'].isin(['M', 'F'])]
df5['year'] = df5['year'].astype(int)

trend = (
    df5.groupby(['year', 'gender'])['finish_min']
    .agg(median='median',
         q25=lambda x: x.quantile(0.25),
         q75=lambda x: x.quantile(0.75))
    .reset_index()
)
participation = df5.groupby('year').size().reset_index(name='n')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

for gender, color, label in [('M', C_MALE, 'Men'), ('F', C_FEMALE, 'Women')]:
    g = trend[trend['gender'] == gender].sort_values('year')
    ax1.plot(g['year'], g['median'], color=color, lw=2.5, marker='o', markersize=4, label=label)
    ax1.fill_between(g['year'], g['q25'], g['q75'], alpha=0.12, color=color)
    for idx, dy in [(0, -16), (-1, -16)]:
        val = g.iloc[idx]['median']
        yr  = g.iloc[idx]['year']
        ax1.annotate(f"{int(val//60)}h{int(val%60):02d}m",
                     xy=(yr, val), xytext=(yr, val + dy),
                     color=color, fontsize=9, ha='center')

ax1.set_title('Median Finish Times Stable Over 20 Years Despite Surging Participation\n(shaded band = interquartile range)')
ax1.set_ylabel('Median Finish Time')
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x//60)}h{int(x%60):02d}m'))
ax1.legend()

ax2.bar(participation['year'], participation['n'] / 1000, color=C_LIGHT, alpha=0.9)
ax2.set_ylabel('Finishers (k)')
ax2.set_xlabel('Year')
ax2.set_xticks(range(2000, 2020, 2))

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '05_year_trend.png'), dpi=150, bbox_inches='tight')
plt.show()


**Design justification (Viz 5):**
- *Small multiples (linked axes)*: the two panels (trend + participation) share an X-axis so the viewer can read both simultaneously without re-orienting — Gestalt continuity across panels.
- *IQR shaded band*: shows the spread of the distribution, not just the median. A single line would hide whether the distribution has narrowed or widened over time (Tufte: graphical integrity).
- *Bar chart for participation count* (lower panel): counts are discrete and non-negative — a bar chart is more honest than a line for count data (Stephen Few: choose chart type to match data semantics).
- *Direct annotation of first and last values*: gives the viewer the answer (net change over 20 years) without requiring them to read the Y-axis twice and subtract (reduces working memory load).


---
## Visualization 6 — Training Volume vs. Speed
**Research question:** Does training volume (km/week) correlate with performance?


In [ ]:
df6 = master.dropna(subset=['strava_weekly_km', 'finish_min']).copy()
df6 = df6[df6['strava_weekly_km'].between(5, 200)]

df6['km_bin']   = (df6['strava_weekly_km'] // 10 * 10).astype(int)
df6['time_bin'] = (df6['finish_min'] // 10 * 10).astype(int)

bubble = (
    df6.groupby(['km_bin', 'time_bin'])
    .size()
    .reset_index(name='count')
)

slope6, intercept6, r6, p6, _ = stats.linregress(df6['strava_weekly_km'], df6['finish_min'])
xr6 = np.linspace(df6['strava_weekly_km'].min(), df6['strava_weekly_km'].max(), 100)

fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(
    bubble['km_bin'], bubble['time_bin'],
    s=bubble['count'] / bubble['count'].max() * 800,
    c=bubble['time_bin'], cmap='Blues_r',
    alpha=0.7, linewidths=0.3, edgecolors=C_LIGHT,
)
plt.colorbar(sc, ax=ax, label='Finish time bin (min)')
ax.plot(xr6, slope6 * xr6 + intercept6, color=C_FEMALE, lw=2,
        label=f'Trend  r={r6:.2f}')

for n_label in [100, 500, 2000]:
    ax.scatter([], [], s=n_label / bubble['count'].max() * 800,
               c=C_LIGHT, label=f'{n_label} runners', alpha=0.7)
ax.legend(title='Bubble size')

ax.set_title('Weekly Training Volume vs. Finish Time\n(Strava data, group-level medians)')
ax.set_xlabel('Weekly Training Volume (km/week)  [Strava median per gender × age group × race]')
ax.set_ylabel('Finish Time')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x//60)}h{int(x%60):02d}m'))

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '06_training_bubble.png'), dpi=150, bbox_inches='tight')
plt.show()

**Design justification (Viz 6):**
- *Bubble area ∝ count* (not radius): Tufte graphical integrity — if area encodes quantity, it must scale with area. Using radius would inflate the visual impression of large groups by π.
- *Binning into grid cells*: reduces 36K individual points to a readable bubble map while preserving density information — Tufte's principle of showing many numbers in small space.
- *Sequential colormap on time bins*: the same Blues_r used in Viz 4 maintains the project-wide convention that darker = faster. Consistent palette across charts reduces working memory load (Lezione 3a).
- *Legend for bubble size*: three reference bubbles anchor the visual encoding — without them the size mapping is ambiguous.

---
## Visualization 7 — Distribution of Finish Times
**Research question:** What is the overall distribution of marathon finish times, and how do men and women differ?


In [ ]:
df7 = master.dropna(subset=['finish_min', 'gender']).copy()
df7 = df7[df7['gender'].isin(['M', 'F'])]

def fd_bins(data, range_min, range_max):
    iqr = np.percentile(data, 75) - np.percentile(data, 25)
    bw  = 2 * iqr / len(data) ** (1/3)
    return np.arange(range_min, range_max + bw, bw)

t_min, t_max = 120, 600
bins = fd_bins(df7['finish_min'].values, t_min, t_max)

fig, ax = plt.subplots(figsize=(11, 6))

for gender, color, label in [('M', C_MALE, 'Men'), ('F', C_FEMALE, 'Women')]:
    sub = df7[df7['gender'] == gender]['finish_min'].values
    ax.hist(sub, bins=bins, density=True, color=color, alpha=0.4, label=f'{label} (n={len(sub):,})')
    kde = stats.gaussian_kde(sub, bw_method='scott')
    xk  = np.linspace(t_min, t_max, 400)
    ax.plot(xk, kde(xk), color=color, lw=2.5)
    med = np.median(sub)
    ax.axvline(med, color=color, lw=1, linestyle='--', alpha=0.7)
    ax.text(med + 2, ax.get_ylim()[1] * 0.85,
            f'Median\n{int(med//60)}h{int(med%60):02d}m',
            color=color, fontsize=9)

ax.set_title('Distribution of Marathon Finish Times by Gender')
ax.set_xlabel('Finish Time')
ax.set_ylabel('Density')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x//60)}h{int(x%60):02d}m'))
ax.set_xlim(t_min, t_max)
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '07_finish_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


**Design justification (Viz 7):**
- *KDE overlay on histogram* (Stephen Few: use density curves to reveal the true shape): the histogram bars quantize the distribution; the KDE reveals the underlying continuous shape. Together they show both raw counts and smooth form.
- *Freedman–Diaconis bin width*: data-driven bin selection avoids the arbitrary choices that create misleading spike/gap artifacts (Stephen Few: bin width should reflect the data, not the default).
- *Alpha=0.4 on overlapping histograms*: Gestalt figure/ground — each series reads as its own layer, and the overlap area is visible rather than hidden.
- *Dashed median lines with direct annotation*: removes the need to read exact axis tick values — reduces working memory load.
- *Density (not count) on Y-axis*: makes the two gender groups comparable despite different n sizes.

---
## Summary — Correlation Comparison Across Predictors

In [ ]:
predictors = {
    'Weekly training (km)': 'strava_weekly_km',
    'Age':                  'age',
    'Temperature (°C)':     'temp_mean_c',
    'Wind speed (km/h)':    'wind_kmh',
    'Precipitation (mm)':   'precip_mm',
}

corrs = []
for label, col in predictors.items():
    if col in master.columns:
        sub = master[['finish_min', col]].dropna()
        if len(sub) > 50:
            r, p = stats.pearsonr(sub[col], sub['finish_min'])
            corrs.append({'Predictor': label, 'r': r, '|r|': abs(r), 'p': p})

corr_df = pd.DataFrame(corrs).sort_values('|r|', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = [C_MALE if r < 0 else C_FEMALE for r in corr_df['r']]
ax.barh(corr_df['Predictor'], corr_df['r'], color=colors)
ax.axvline(0, color=C_BOTH, lw=1)
ax.set_title('Correlation of Each Factor with Finish Time\n(negative r = associated with faster times)')
ax.set_xlabel('Pearson r')
ax.invert_yaxis()

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=C_MALE,   label='Negative correlation (faster)'),
    Patch(color=C_FEMALE, label='Positive correlation (slower)'),
], loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '08_predictor_correlations.png'), dpi=150, bbox_inches='tight')
plt.show()
print(corr_df[['Predictor', 'r', 'p']].to_string(index=False))
